# Auslesen der SQLite Metadaten

In [1]:
import sqlite3 as sl

import pandas as pd

con = sl.connect('spotify.sqlite')

with con:
    df_master = pd.read_sql_query("SELECT * FROM sqlite_master", con)
    print(f"Anzahl der Tabellen und Sichten: {len(df_master)} \n")
    print("Info über die Tabellen und Sichten:")
    df_master.info()


Anzahl der Tabellen und Sichten: 9 

Info über die Tabellen und Sichten:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   type      9 non-null      object
 1   name      9 non-null      object
 2   tbl_name  9 non-null      object
 3   rootpage  9 non-null      int64 
 4   sql       9 non-null      object
dtypes: int64(1), object(4)
memory usage: 488.0+ bytes


Da es sich um eine Spotify-Datenbank handelt, sind die wichtigsten Tabellen wahrscheinlich diejenigen, die Informationen über Songs, Alben, Künstler und Playlists enthalten. Um diese Tabellen zu identifizieren, können wir uns die Namen der Tabellen in der Datenbank ansehen. Es gibt die Spalte "name" und die Spalte "tbl_name", die beide die Namen der Tabellen enthalten. Es ist nicht ganz klar, was der Unterschied zwischen diesen beiden Spalten ist, daher sehen wir uns beide an.

In [2]:
print(f"Spalte name: \n{df_master.name} \n")
print(f"Spalte tbl_name: \n{df_master.tbl_name}")


Spalte name: 
0              albums
1             artists
2      audio_features
3              genres
4    r_albums_artists
5     r_albums_tracks
6      r_artist_genre
7      r_track_artist
8              tracks
Name: name, dtype: object 

Spalte tbl_name: 
0              albums
1             artists
2      audio_features
3              genres
4    r_albums_artists
5     r_albums_tracks
6      r_artist_genre
7      r_track_artist
8              tracks
Name: tbl_name, dtype: object


Die Spalte "name" und die Spalte "tbl_name" scheinen in diesem Fall identisch zu sein. Wir können also eine der beiden Spalten verwenden, um die Namen der Tabellen in der Datenbank zu erhalten. Wir filtern das Dataframe, um nur die Einträge mit dem Typ "table" zu erhalten, da wir nur an den Tabellen interessiert sind.

In [3]:
df_tables = df_master[df_master['type'] == 'table']
print(f"Tabellen in der Datenbank: \n{df_tables[['name', 'tbl_name']]}")

Tabellen in der Datenbank: 
               name          tbl_name
0            albums            albums
1           artists           artists
2    audio_features    audio_features
3            genres            genres
4  r_albums_artists  r_albums_artists
5   r_albums_tracks   r_albums_tracks
6    r_artist_genre    r_artist_genre
7    r_track_artist    r_track_artist
8            tracks            tracks


Fangen wir damit an, uns die Tabelle "tracks" anzusehen, da sie wahrscheinlich Informationen über die Songs enthält.

In [4]:
con.text_factory = lambda b: b.decode(errors = 'ignore')
df_tracks = pd.read_sql_query("SELECT * FROM tracks", con)

print("Info über die Tabelle 'tracks':")
df_tracks.info()
print()

print("Beschreibung der Tabelle 'tracks':")
print(df_tracks.describe(include='all'))
print()

print("Erste 5 Einträge der Tabelle 'tracks':")
print(df_tracks.head())


Info über die Tabelle 'tracks':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8741672 entries, 0 to 8741671
Data columns (total 10 columns):
 #   Column            Dtype  
---  ------            -----  
 0   id                object 
 1   disc_number       int64  
 2   duration          int64  
 3   explicit          int64  
 4   audio_feature_id  object 
 5   name              object 
 6   preview_url       object 
 7   track_number      int64  
 8   popularity        int64  
 9   is_playable       float64
dtypes: float64(1), int64(5), object(4)
memory usage: 666.9+ MB

Beschreibung der Tabelle 'tracks':
                            id   disc_number      duration      explicit  \
count                  8741672  8.741672e+06  8.741672e+06  8.741672e+06   
unique                 8741672           NaN           NaN           NaN   
top     1dizvxctg9dHEyaYTFufVi           NaN           NaN           NaN   
freq                         1           NaN           NaN           NaN   
mea

Wir sehen, dass die Tabelle "tracks" 10 Spalten hat:
- id: Die eindeutige ID des Songs
- disc_number: Vermutlich die Nummer der Disc, auf der sich der Song befindet
- duration: Die Dauer des Songs (in Millisekunden, da der Median 216160 beträgt, was 3,6 Minuten entspricht)
- explicit: Ein Indikator dafür, ob der Song explizite Inhalte enthält (0 = nein, 1 = ja)
- audio_feature_id: Die ID der Audio-Features des Songs
- name: Der Name des Songs
- preview_url: Die URL zur Vorschau des Songs
- track_number: Vermutlich die Nummer des Songs auf der Disc
- popularity: Die Popularität des Songs (auf einer Skala von 0 bis 100)
- is_playable: Noch nicht klar, was diese Spalte bedeutet. Die Werten sind 0, 1 oder NaN.

Als nächstes lesen wir auch die anderen Tabellen aus der Datenbank ein, um einen Überblick über die verfügbaren Daten zu bekommen. Wir geben uns jeweils die Info, Beschreibung und die ersten 5 Einträge der Tabellen aus.

In [5]:
df_artists = pd.read_sql_query("SELECT * FROM artists", con)
print("Info über die Tabelle 'artists':")
df_artists.info()
print()
print("Beschreibung der Tabelle 'artists':")
print(df_artists.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'artists':")
print(df_artists.head())

df_albums = pd.read_sql_query("SELECT * FROM albums", con)
print("Info über die Tabelle 'albums':")
df_albums.info()
print()
print("Beschreibung der Tabelle 'albums':")
print(df_albums.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'albums':")
print(df_albums.head())

df_audio_features = pd.read_sql_query("SELECT * FROM audio_features", con)
print("Info über die Tabelle 'audio_features':")
df_audio_features.info()
print()
print("Beschreibung der Tabelle 'audio_features':")
print(df_audio_features.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'audio_features':")
print(df_audio_features.head())

df_genres = pd.read_sql_query("SELECT * FROM genres", con)
print("Info über die Tabelle 'genres':")
df_genres.info()
print()
print("Beschreibung der Tabelle 'genres':")
print(df_genres.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'genres':")
print(df_genres.head())

df_r_ablums_artists = pd.read_sql_query("SELECT * FROM r_albums_artists", con)
print("Info über die Tabelle 'r_albums_artists':")
df_r_ablums_artists.info()
print()
print("Beschreibung der Tabelle 'r_albums_artists':")
print(df_r_ablums_artists.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'r_albums_artists':")
print(df_r_ablums_artists.head())

df_r_albums_tracks = pd.read_sql_query("SELECT * FROM r_albums_tracks", con)
print("Info über die Tabelle 'r_albums_tracks':")
df_r_albums_tracks.info()
print()
print("Beschreibung der Tabelle 'r_albums_tracks':")
print(df_r_albums_tracks.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'r_albums_tracks':")
print(df_r_albums_tracks.head())

df_r_artist_genre = pd.read_sql_query("SELECT * FROM r_artist_genre", con)
print("Info über die Tabelle 'r_artist_genre':")
df_r_artist_genre.info()
print()
print("Beschreibung der Tabelle 'r_artist_genre':")
print(df_r_artist_genre.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'r_artist_genre':")
print(df_r_artist_genre.head())

df_r_track_artist = pd.read_sql_query("SELECT * FROM r_track_artist", con)
print("Info über die Tabelle 'r_track_artist':")
df_r_track_artist.info()
print()
print("Beschreibung der Tabelle 'r_track_artist':")
print(df_r_track_artist.describe(include='all'))
print()
print("Erste 5 Einträge der Tabelle 'r_track_artist':")
print(df_r_track_artist.head())





Info über die Tabelle 'artists':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066031 entries, 0 to 1066030
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   name        1066031 non-null  object
 1   id          1066031 non-null  object
 2   popularity  1066031 non-null  int64 
 3   followers   1066031 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 32.5+ MB

Beschreibung der Tabelle 'artists':
           name                      id    popularity     followers
count   1066031                 1066031  1.066031e+06  1.066031e+06
unique  1011159                 1066031           NaN           NaN
top       Ghost  4tujQJicOnuZRLiBFdp3Ou           NaN           NaN
freq         27                       1           NaN           NaN
mean        NaN                     NaN  5.417925e+00  7.012360e+03
std         NaN                     NaN  1.194380e+01  2.087184e+05
min         NaN                     NaN